# Does reward hacking show up inside the model before it shows up in behaviour?

Trains a small coder model with RL (GRPO) on tasks whose grader can be gamed, plus a control run whose grader
can't, and then reads every checkpoint from the inside.

**To run:** **Runtime → Change runtime type → A100 GPU** (an L4 or T4 works with the 0.5B model, more slowly),
then **Runtime → Run all**. Expect a few hours. Keep the tab open.

When it finishes, copy the results text at the bottom back to Claude.

In [ ]:
# 1. Get the code
!rm -rf safeinterp && git clone -q -b claude/gifted-darwin-ithng2 https://github.com/nyancatspace/safeinterp
%cd safeinterp
!pip install -q -e ".[plot,misalign]" 2>&1 | tail -1

In [ ]:
# 2. Settings, picked from the GPU's memory
import torch
assert torch.cuda.is_available(), "needs a GPU: Runtime -> Change runtime type"
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
BIG = mem > 30
MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct" if BIG else "Qwen/Qwen2.5-Coder-0.5B-Instruct"
LAYERS = "14,21" if BIG else "12,18"
STEPS, SAVE_EVERY = 150, 5
MICRO = 8 if BIG else 2  # sequences per backward pass; lower it if you run out of memory
HINT = "weak"  # tells the model the grader only checks the exit status, so hacks are found within a short run
print(torch.cuda.get_device_name(0), f"{mem:.0f} GB ->", MODEL, "layers", LAYERS)

In [ ]:
# 3. RL with the hackable grader (the slow step)
!python -m safeinterp rlhack-train --model {MODEL} --steps {STEPS} --save-every {SAVE_EVERY} --micro-batch {MICRO} --hint {HINT} --grader naive --out runs/naive 2>&1 | grep --line-buffered "^step"

In [ ]:
# 4. Control: same RL, but the grader also runs hidden tests, so hacks don't pay
!python -m safeinterp rlhack-train --model {MODEL} --steps {STEPS} --save-every {SAVE_EVERY} --micro-batch {MICRO} --hint {HINT} --grader honest --out runs/honest 2>&1 | grep --line-buffered "^step"

In [ ]:
# 5. SAEs on the starting model, trained on both runs' rollouts
!python -m safeinterp train --model {MODEL} --layers {LAYERS} --hook resid_post --tokens 4M --ctx-len 256 --buffer-tokens 131072 --batch-size 2048 --expansion 8 --text-files "runs/*/rollouts_text/*.txt" --out runs/sae 2>&1 | grep --line-buffered "recovered\|data exhausted"

In [ ]:
# 6. Monitor both runs (control first, so the main report can show its false alarms)
!python -m safeinterp rlhack-monitor --run runs/honest --sae-dir runs/sae --out reports/honest 2>&1 | tail -3
!python -m safeinterp rlhack-monitor --run runs/naive --sae-dir runs/sae --control reports/honest/monitor.json --out reports/naive 2>&1 | tail -3

In [ ]:
# 7. Plot
from IPython.display import Image, display
display(Image("reports/naive/monitor.png"))

In [ ]:
# 8. Results: copy everything below and paste it to Claude
import json
print(open("reports/naive/report.md").read())
for name in ("naive", "honest"):
    rows = json.load(open(f"runs/{name}/hack_rate.json"))
    print(f"\n## {name}: RL rollouts every 10 steps")
    for r in rows[::10] + rows[-1:]:
        print({k: r[k] for k in ("step", "reward", "hidden_pass", "hack_rate", "gap", "flag_rate", "hack_hardcode", "hack_exit", "hack_always_equal", "mean_len")})
print("\n## monitor rows (naive)")
for r in json.load(open("reports/naive/monitor.json"))["rows"]:
    print(r)
!for f in runs/sae/layer_*/eval.json; do echo $f; cat $f; done